In [1]:
pip install chromadb sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [4]:
from sentence_transformers import SentenceTransformer
import chromadb

In [11]:

documents = [
    "Jungles are the lungs of our planet, providing the oxygen we breathe and regulating the global climate.",
    "Protecting rainforests preserves biodiversity, safeguarding millions of unique plant and animal species from extinction.",
    "Street animals deserve compassion and care, as they brave harsh weather and hunger every single day.",
    "Deforestation drives climate change, making the preservation of our green canopies more urgent than ever.",
    "Spaying and neutering street animals is a humane way to control their population and reduce homelessness.",
    "Small acts of kindness, like leaving out clean water and food, can save a stray animal's life.",
    "When we destroy jungles, we lose natural habitats and disrupt the delicate balance of Earth's ecosystems.",
    "Deforestation drives climate change, making the preservation of our green canopies more urgent than ever.",
    "Street animals are not save.",
]

In [6]:
# Load embedding model ----------
# Turns text into vectors (lists of numbers)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Create vector database ----------
client = chromadb.Client()                 # in-memory database
collection = client.create_collection(name="street_animals")

# Index: embed and store ----------
embeddings = embedder.encode(documents).tolist()

collection.add(
    ids=[f"doc_{i}" for i in range(len(documents))],
    documents=documents,
    embeddings=embeddings,
)
print(f"Stored {collection.count()} documents.\n")

#  Retrieve ----------
def retrieve(question, k=2):
    query_embedding = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    return results["documents"][0]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 942.90it/s]


Stored 9 documents.



In [12]:
# Build the prompt ----------
def build_prompt(question):
    chunks = retrieve(question)
    context = "\n".join(f"- {c}" for c in chunks)
    return f"""Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}
Answer:"""

# ---------- 7. Try it ----------
question = "Is Street Animal are safe ?"
prompt = build_prompt(question)
print(prompt)

Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
- Street animals are not save.
- Street animals deserve compassion and care, as they brave harsh weather and hunger every single day.

Question: Is Street Animal are safe ?
Answer:


In [10]:
pip install anthropic

Note: you may need to restart the kernel to use updated packages.


In [7]:
import anthropic

In [8]:
llm = anthropic.Anthropic(api_key="Your_API_KEY")

response = llm.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=300,
    messages=[{"role": "user", "content": prompt}],
)

print(response.content[0].text)

/tmp/ipykernel_21815/1055526133.py:3: DeprecationWarning: The model 'claude-sonnet-4-5' is deprecated and will reach end-of-life on November 30th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = llm.messages.create(


AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': None}

In [11]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


In [12]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import pipeline
import ollama

/usr/local/python/3.14.2/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")
result = generator(prompt, max_new_tokens=100)
print(result[0]["generated_text"])

KeyError: 'The task text2text-generation has been removed from transformers. Use the `text-generation` pipeline with a chat model instead. See the migration guide for more details: https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md#pipelines'

In [13]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",   # small, free, ~1 GB
)

messages = [{"role": "user", "content": prompt}]

result = generator(messages, max_new_tokens=100)
print(result[0]["generated_text"][-1]["content"])

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 10858.89it/s]
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Street Animals are not saved.


In [ ]:
response = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": prompt}],
)

print(response["message"]["content"])

ResponseError: llama-server process has terminated: signal: terminated (status code: 500)